# Lab 13 · Reference solution

The polished final implementation of [Lab 13: Multi-agent RAG from scratch](../README.md).

The integrative lab. Composes Path 02's retrieval pipeline (Labs 06-08: dense + BM25 + RRF + cross-encoder rerank, with optional contextual augmentation when Lab 08's cache is available) with Lab 10's supervisor-worker pattern. Retriever-worker wraps the pipeline; supervisor decides when to retrieve via four retrieval-decision rules; synthesizer composes with chunk_id citations.

This notebook is the reference implementation; refer to [`../lab.ipynb`](../lab.ipynb) for the pedagogical step-by-step build. The [`solution README`](./README.md) covers implementation choices, common variations, and bugs to watch for.

## Setup

Locate the corpus (Lab 06's directory). Auto-detect Lab 08's context cache.

In [ ]:
import hashlib
import json
import os
import pathlib
import re
import warnings
from dataclasses import dataclass, field
from typing import Any

from dotenv import load_dotenv
from pydantic import BaseModel, ConfigDict, Field

here = pathlib.Path.cwd()
REPO_ROOT = None
for parent in [here, *here.parents]:
    if (parent / ".env.example").exists():
        load_dotenv(parent / ".env")
        REPO_ROOT = parent
        break

assert os.getenv("OPENAI_API_KEY") or os.getenv("ANTHROPIC_API_KEY")

PROVIDER = "openai"
MODEL = {"openai": "gpt-4o-mini", "anthropic": "claude-haiku-4-5-20251001"}[PROVIDER]

# Locate corpus (Lab 06's directory) and Lab 08's context cache
CORPUS_DIR = REPO_ROOT / "labs/06-agentic-rag-from-scratch/corpus"
assert CORPUS_DIR.exists(), f"Corpus missing at {CORPUS_DIR}"

CONTEXT_CACHE_PATH = REPO_ROOT / "labs/08-contextual-retrieval-and-query-rewriting/context_cache.json"
HAS_CONTEXT_CACHE = CONTEXT_CACHE_PATH.exists()

print(f"Using {PROVIDER} / {MODEL}")
print(f"Corpus: {CORPUS_DIR}")
print(f"Context cache: {'found — v3 candidate' if HAS_CONTEXT_CACHE else 'not found — v2 only'}")


## Lab 10 machinery (chat client + helpers)

In [ ]:
@dataclass
class ToolCall:
    id: str
    name: str
    arguments: dict


@dataclass
class AssistantMessage:
    content: str | None
    tool_calls: list[ToolCall] = field(default_factory=list)


def chat_with_tools(messages: list[dict], tools: list[dict] | None = None,
                     tool_choice: str = "auto", temperature: float = 0) -> AssistantMessage:
    if PROVIDER == "openai":
        from openai import OpenAI
        resp = OpenAI().chat.completions.create(
            model=MODEL, messages=messages, tools=tools,
            tool_choice=tool_choice if tools else None, temperature=temperature,
        )
        msg = resp.choices[0].message
        return AssistantMessage(
            content=msg.content,
            tool_calls=[
                ToolCall(id=tc.id, name=tc.function.name,
                         arguments=json.loads(tc.function.arguments))
                for tc in (msg.tool_calls or [])
            ],
        )
    elif PROVIDER == "anthropic":
        from anthropic import Anthropic
        client = Anthropic()
        system = next((m["content"] for m in messages if m["role"] == "system"), "")
        non_system = [m for m in messages if m["role"] != "system"]
        anth_tools = [
            {"name": t["function"]["name"], "description": t["function"]["description"],
             "input_schema": t["function"]["parameters"]}
            for t in (tools or [])
        ]
        resp = client.messages.create(
            model=MODEL, system=system, messages=non_system,
            tools=anth_tools or None, max_tokens=2048, temperature=temperature,
        )
        text = "".join(b.text for b in resp.content if hasattr(b, "text"))
        tcs = [ToolCall(id=b.id, name=b.name, arguments=dict(b.input))
               for b in resp.content if getattr(b, "type", None) == "tool_use"]
        return AssistantMessage(content=text or None, tool_calls=tcs)
    raise ValueError(f"Unknown PROVIDER: {PROVIDER!r}")


def _action_hash(name: str, args: dict) -> str:
    return hashlib.sha256(
        (name + "|" + json.dumps(args, sort_keys=True)).encode()
    ).hexdigest()[:16]


class StrictModel(BaseModel):
    model_config = ConfigDict(extra="forbid")


## Chunker (inline, matches Lab 06's parameters)

`TARGET_TOKENS=160`, `OVERLAP_TOKENS=32` — pinned for Lab 08 cache compatibility.

In [ ]:
TARGET_TOKENS = 160
OVERLAP_TOKENS = 32


def approx_tokens(text: str) -> int:
    return int(len(text.split()) / 0.75)


def split_at_paragraphs(text: str) -> list[str]:
    parts = re.split(r"\n\s*\n", text)
    return [p.strip() for p in parts if p.strip()]


def split_at_sentences(text: str) -> list[str]:
    parts = re.split(r"(?<=[.!?])\s+", text)
    return [p.strip() for p in parts if p.strip()]


def chunk_text(text: str, source: str, title: str) -> list[dict]:
    chunks: list[dict] = []
    chunk_idx = 0

    def emit(content: str) -> None:
        nonlocal chunk_idx
        if not content.strip():
            return
        chunks.append({
            "chunk_id": f"{pathlib.Path(source).stem}_{chunk_idx:03d}",
            "text": content.strip(), "source": source, "title": title,
        })
        chunk_idx += 1

    current = ""
    for para in split_at_paragraphs(text):
        candidate = (current + "\n\n" + para) if current else para
        if approx_tokens(candidate) <= TARGET_TOKENS:
            current = candidate
            continue
        if current:
            emit(current)
            tail_words = current.split()[-OVERLAP_TOKENS:]
            current = " ".join(tail_words) + "\n\n" + para
        else:
            buf = ""
            for s in split_at_sentences(para):
                cand = (buf + " " + s) if buf else s
                if approx_tokens(cand) <= TARGET_TOKENS:
                    buf = cand
                else:
                    if buf:
                        emit(buf)
                    buf = s
            current = buf
    if current:
        emit(current)
    return chunks


def load_all_chunks(corpus_dir: pathlib.Path) -> list[dict]:
    chunks: list[dict] = []
    for md_file in sorted(corpus_dir.glob("*.md")):
        if md_file.name == "README.md":
            continue
        text = md_file.read_text()
        title_match = re.search(r"^#\s+(.+)$", text, re.M)
        title = title_match.group(1).strip() if title_match else md_file.stem
        chunks.extend(chunk_text(text, source=md_file.name, title=title))
    return chunks


ALL_CHUNKS = load_all_chunks(CORPUS_DIR)
print(f"Loaded {len(ALL_CHUNKS)} chunks")


## Retrieval pipeline (v2 inline; v3 if cache available with ≥90% coverage)

Dense + BM25 + RRF + cross-encoder rerank. v3 adds contextual augmentation as an indexing-only intervention — the retriever returns the ORIGINAL chunk text either way.

In [ ]:
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
from sentence_transformers import SentenceTransformer, CrossEncoder
from rank_bm25 import BM25Okapi


DENSE_MODEL = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
CROSS_ENCODER = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

# v3 auto-detect with 90% coverage check (prevents silent parameter-drift corruption)
CONTEXT_CACHE: dict[str, str] = {}
if HAS_CONTEXT_CACHE:
    try:
        candidate = json.loads(CONTEXT_CACHE_PATH.read_text())
        our_ids = {c["chunk_id"] for c in ALL_CHUNKS}
        overlap = set(candidate.keys()) & our_ids
        coverage = len(overlap) / len(our_ids) if our_ids else 0
        if coverage >= 0.9:
            CONTEXT_CACHE = candidate
            print(f"Using v3 (contextual): cache covers {coverage:.0%} of chunks")
        else:
            print(f"WARNING: cache covers only {coverage:.0%}; falling back to v2")
    except Exception as e:
        print(f"WARNING: cache load failed ({e}); falling back to v2")

PIPELINE = "v3-contextual" if CONTEXT_CACHE else "v2"


def _indexed_text(chunk: dict) -> str:
    """Text used for indexing. v3 prepends context; v2 returns chunk text as-is."""
    if CONTEXT_CACHE and chunk["chunk_id"] in CONTEXT_CACHE:
        return CONTEXT_CACHE[chunk["chunk_id"]] + "\n\n" + chunk["text"]
    return chunk["text"]


INDEXED_TEXTS = [_indexed_text(c) for c in ALL_CHUNKS]
DENSE_INDEX = DENSE_MODEL.encode(INDEXED_TEXTS, convert_to_numpy=True,
                                   normalize_embeddings=True, show_progress_bar=False)
TOKENIZED_CHUNKS = [t.lower().split() for t in INDEXED_TEXTS]
BM25 = BM25Okapi(TOKENIZED_CHUNKS)

print(f"Pipeline: {PIPELINE}, dense index: {DENSE_INDEX.shape}, "
      f"BM25 corpus: {len(TOKENIZED_CHUNKS)} chunks")


def _rrf_merge(dense_ranks: list[int], bm25_ranks: list[int], k: int = 60) -> list[int]:
    scores: dict[int, float] = {}
    for rank, idx in enumerate(dense_ranks, start=1):
        scores[idx] = scores.get(idx, 0) + 1 / (k + rank)
    for rank, idx in enumerate(bm25_ranks, start=1):
        scores[idx] = scores.get(idx, 0) + 1 / (k + rank)
    return [idx for idx, _ in sorted(scores.items(), key=lambda kv: -kv[1])]


def retrieve_chunks(query: str, top_k: int = 5, candidate_k: int = 30) -> list[dict]:
    """Lab 07's v2 pipeline: dense + BM25 + RRF + cross-encoder rerank."""
    query_vec = DENSE_MODEL.encode([query], convert_to_numpy=True,
                                     normalize_embeddings=True,
                                     show_progress_bar=False)[0]
    dense_scores = DENSE_INDEX @ query_vec
    dense_ranks = np.argsort(-dense_scores)[:candidate_k].tolist()

    bm25_scores = BM25.get_scores(query.lower().split())
    bm25_ranks = np.argsort(-bm25_scores)[:candidate_k].tolist()

    merged_ranks = _rrf_merge(dense_ranks, bm25_ranks)[:candidate_k]

    pairs = [(query, INDEXED_TEXTS[idx]) for idx in merged_ranks]
    rerank_scores = CROSS_ENCODER.predict(pairs, show_progress_bar=False)
    reranked = sorted(zip(merged_ranks, rerank_scores, strict=False),
                       key=lambda kv: -kv[1])

    # Return ORIGINAL chunk text (not indexed text — same as Lab 08)
    results = []
    for idx, score in reranked[:top_k]:
        c = ALL_CHUNKS[idx]
        results.append({
            "id": c["chunk_id"], "text": c["text"],
            "source": c["source"], "title": c["title"],
            "score": float(score),
        })
    return results


## Retriever worker

Wraps the pipeline as a single function. Returns structured envelope; the `score` field is informational only — the supervisor's system prompt is told not to reason about it.

In [ ]:
MIN_RERANK_SCORE = -2.0  # Empirical floor for this corpus + cross-encoder model


def retriever_agent(query: str, top_k: int = 5) -> dict:
    """Run retrieval. Return {status, chunks, query, pipeline}.

    Status values:
    - "ok": at least one chunk above MIN_RERANK_SCORE
    - "empty": no chunks above floor (corpus doesn't have it)
    - "error": pipeline failure
    """
    try:
        chunks = retrieve_chunks(query, top_k=top_k)
    except Exception as e:
        return {"status": "error", "kind": "pipeline_error",
                "detail": f"{type(e).__name__}: {e}",
                "chunks": [], "query": query, "pipeline": PIPELINE}

    usable = [c for c in chunks if c["score"] >= MIN_RERANK_SCORE]
    if not usable:
        top_score = chunks[0]["score"] if chunks else float("-inf")
        return {"status": "empty",
                "detail": (f"no chunks above relevance floor "
                            f"(top score: {top_score:.2f} < {MIN_RERANK_SCORE})"),
                "chunks": [], "query": query, "pipeline": PIPELINE}
    return {"status": "ok", "chunks": usable, "query": query, "pipeline": PIPELINE}


## Synthesizer worker

Cites by `chunk_id`. Refuses to paraphrase beyond what chunks support.

In [ ]:
SYNTHESIZER_SYSTEM_PROMPT = """You are a synthesizer. You receive a user query plus
retrieved chunks from a document corpus.

Your job: produce a clear, accurate answer using ONLY the chunks.

Rules:
1. CITE BY CHUNK_ID. For every claim from a chunk, reference its id inline as
   [chunk_id_here]. Then list citations at the end:

       [chunk_id_1] source — title
       [chunk_id_2] source — title

2. DO NOT INVENT CLAIMS. If chunks don't cover something the user asked, say so.

3. DO NOT PARAPHRASE BEYOND WHAT CHUNKS SUPPORT. You can restate in different
   words; you cannot extrapolate.

4. If you can only address part of the question from chunks, be explicit.
"""


def synthesizer_agent(query: str, chunks: list[dict]) -> dict:
    if not chunks:
        return {"status": "error", "kind": "no_chunks",
                "detail": "synthesizer called with empty chunks list"}

    chunks_block = "\n\n".join(
        f"[{c['id']}] (source: {c['source']}, title: {c['title']})\n{c['text']}"
        for c in chunks
    )
    user_prompt = (
        f"USER QUERY:\n{query}\n\n"
        f"RETRIEVED CHUNKS:\n{chunks_block}\n\n"
        "Compose the answer. Cite by chunk_id inline."
    )
    msg = chat_with_tools(
        [{"role": "system", "content": SYNTHESIZER_SYSTEM_PROMPT},
         {"role": "user", "content": user_prompt}],
        temperature=0,
    )
    return {"status": "ok", "answer": msg.content or "",
            "chunk_ids_available": [c["id"] for c in chunks]}


## Supervisor with four retrieval-decision rules

Two worker tools: `call_retriever` + `call_synthesizer`. Corpus description in system prompt lets the LLM judge whether a query is corpus-grounded. Chunks pass through to the synthesizer VERBATIM (structural mitigation for citation drift).

In [ ]:
SUPERVISOR_MAX_STEPS = 6


class CallRetrieverArgs(StrictModel):
    query: str = Field(description="Focused factual query. NOT the user's whole question.")
    top_k: int = Field(default=5, description="Number of chunks to retrieve (3-8 sensible).")


class CallSynthesizerArgs(StrictModel):
    query: str = Field(description="The user's original question.")
    chunks: list[dict] = Field(
        description=("Chunks envelope from call_retriever, passed through VERBATIM. "
                     "Do NOT re-format, filter, or summarize.")
    )


def _call_retriever_tool(args: CallRetrieverArgs) -> dict:
    return retriever_agent(args.query, top_k=args.top_k)


def _call_synthesizer_tool(args: CallSynthesizerArgs) -> dict:
    return synthesizer_agent(args.query, args.chunks)


SUPERVISOR_TOOLS_REGISTRY: dict = {
    "call_retriever": (
        _call_retriever_tool, CallRetrieverArgs,
        "Search the document corpus. Returns {status, chunks, query, pipeline}. "
        "Use when the query is grounded in the corpus. Do NOT use for "
        "stable-knowledge questions. Do NOT call more than once with the same query.",
    ),
    "call_synthesizer": (
        _call_synthesizer_tool, CallSynthesizerArgs,
        "Compose the answer from retrieved chunks. Returns {status, answer}. "
        "Pass the chunks list from call_retriever EXACTLY as received — do not "
        "filter, re-number, or summarize. The synthesizer cites by chunk_id.",
    ),
}

CORPUS_DESCRIPTION = """The corpus covers core agentic AI concepts from Path 01:
the agent loop, tool design and selection, the ReAct pattern, search vs. retrieval,
embeddings, vector indexes (HNSW, IVF, FAISS), chunking strategies, and citation
tracking. Each document is a markdown explainer ~200-500 words. The corpus does
NOT cover: multi-agent coordination, retrieval evaluation, language-specific tool
SDKs, or post-2024 framework updates.
"""

SUPERVISOR_SYSTEM_PROMPT = f"""You are a supervisor coordinating two workers: a
retriever (searches a document corpus) and a synthesizer (composes answers from
chunks).

CORPUS:
{CORPUS_DESCRIPTION}

DECISION RULES:

Rule 1 — RETRIEVE WHEN GROUNDED IN CORPUS. Compare the query to the corpus
description. If grounded, retrieve.

Rule 2 — DO NOT RETRIEVE FOR STABLE-KNOWLEDGE QUESTIONS. Definitional questions,
historical facts, general programming questions — answer directly from training
data, be honest that you're doing so.

Rule 3 — ONE RETRIEVAL PER DISTINCT FACTUAL QUESTION. If retrieval returns
status='empty', surface "not in corpus" — do NOT retry with reworded queries.

Rule 4 — PASS CHUNKS VERBATIM TO SYNTHESIZER. When you call call_synthesizer,
pass the chunks list unchanged. Do not summarize, filter, or re-format.

WORKFLOW:
- If retrieval is needed: call_retriever → call_synthesizer → return answer.
- If retrieval is NOT needed: answer directly without calling either worker.
- If retriever returns empty: surface "not in corpus."

CRITICAL: When action-hash dedup refuses a repeated call, move on.
"""


def _supervisor_schemas() -> list[dict]:
    return [
        {"type": "function",
         "function": {"name": name, "description": desc,
                      "parameters": args_model.model_json_schema()}}
        for name, (_fn, args_model, desc) in SUPERVISOR_TOOLS_REGISTRY.items()
    ]


def _supervisor_dispatch(call: ToolCall) -> dict:
    if call.name not in SUPERVISOR_TOOLS_REGISTRY:
        return {"status": "error", "kind": "unknown_worker", "tool": call.name,
                "available": list(SUPERVISOR_TOOLS_REGISTRY)}
    fn, args_model, _ = SUPERVISOR_TOOLS_REGISTRY[call.name]
    try:
        return fn(args_model.model_validate(call.arguments))
    except Exception as e:
        return {"status": "error", "kind": "supervisor_dispatch_error",
                "detail": f"{type(e).__name__}: {e}"}


def multi_agent_rag(query: str) -> dict:
    """Top-level entry: supervisor → optional retriever → optional synthesizer."""
    messages: list[dict] = [
        {"role": "system", "content": SUPERVISOR_SYSTEM_PROMPT},
        {"role": "user", "content": query},
    ]
    seen_actions: set[str] = set()
    schemas = _supervisor_schemas()
    retrieved_chunks: list[dict] = []
    did_retrieve = False

    for step in range(1, SUPERVISOR_MAX_STEPS + 1):
        msg = chat_with_tools(messages, tools=schemas)
        entry: dict[str, Any] = {"role": "assistant", "content": msg.content}
        if msg.tool_calls:
            entry["tool_calls"] = [
                {"id": tc.id, "type": "function",
                 "function": {"name": tc.name, "arguments": json.dumps(tc.arguments)}}
                for tc in msg.tool_calls
            ]
        messages.append(entry)

        if not msg.tool_calls:
            return {"answer": msg.content or "", "did_retrieve": did_retrieve,
                    "retrieved_chunks": retrieved_chunks, "steps": step}

        for tc in msg.tool_calls:
            ah = _action_hash(tc.name, tc.arguments)
            if ah in seen_actions:
                tool_result = {"status": "error", "kind": "repeated_action",
                               "detail": f"Already called {tc.name} with these args."}
            else:
                seen_actions.add(ah)
                tool_result = _supervisor_dispatch(tc)
                if tc.name == "call_retriever":
                    did_retrieve = True
                    if tool_result.get("status") == "ok":
                        retrieved_chunks = tool_result.get("chunks", [])
            messages.append({"role": "tool", "tool_call_id": tc.id,
                             "content": json.dumps(tool_result)[:6000]})

    return {"answer": "[supervisor hit step cap]", "did_retrieve": did_retrieve,
            "retrieved_chunks": retrieved_chunks, "steps": SUPERVISOR_MAX_STEPS}


## Demo

In [ ]:
query = "How does the ReAct pattern differ from a basic agent loop?"
result = multi_agent_rag(query)

print(f"did_retrieve: {result['did_retrieve']}, "
      f"chunks: {len(result['retrieved_chunks'])}, "
      f"supervisor steps: {result['steps']}")
print("=" * 70)
print(result["answer"])


**Sample output (LLM responses will vary; trajectory should be stable):**

```
did_retrieve: True, chunks: 5, supervisor steps: 3
======================================================================
The ReAct pattern extends the basic agent loop by making the reasoning
step explicit [03-react-pattern_000]. In a basic agent loop, the agent
iterates think → act → observe with the "think" step often implicit in
the LLM's tool-call decision. ReAct makes the reasoning explicit:
at each iteration, the agent emits a thought, then an action, then
observes the result [03-react-pattern_001]...

[03-react-pattern_000] 03-react-pattern.md — The ReAct Pattern
[03-react-pattern_001] 03-react-pattern.md — The ReAct Pattern
```

Three supervisor steps: call_retriever → call_synthesizer → finalize. Citations are chunk_ids (not [1]/[2]) — the chain of custody traces back to specific chunks in the corpus.

## Production readiness — out of scope here

For a real deployment you'd want: vector DB integration (Qdrant/Pinecone/Weaviate) instead of in-memory NumPy + BM25; sharded retrieval across multiple corpora; per-query budget enforcement; structured logging at every handoff; circuit breakers when retrieval consistently returns empty; eval-time monitoring of retrieve-skip-rate; the Lab 11 critic-on-synthesis composition for groundedness; the Lab 12 planner composition for compound queries.

This concludes Path 03's Module 1-4 from-scratch material. Module 5 (framework bridge, future) will re-implement Labs 10-13 in LangGraph's multi-agent primitives — the pedagogical payoff for the from-scratch discipline.